In [1]:
import os
import pandas as pd
from pathlib import Path
import requests
from dotenv import load_dotenv
load_dotenv()

DEMAND_PATH = Path("../Energy_demand")

demand = pd.read_csv(DEMAND_PATH / "demanddata_2025.csv")

demand["SETTLEMENT_DATE"] = pd.to_datetime(demand["SETTLEMENT_DATE"],errors= "coerce", format = "%Y-%m-%d")
demand

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,IFA_FLOW,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW
0,2025-01-01,1,21036,26215,19918,5405,6606,0,19270,0,...,-1518,-1026,1,-232,-386,-558,96,0,241,0
1,2025-01-01,2,21222,26063,20167,5358,6606,0,19270,0,...,-1513,-1023,1,-124,-282,-564,51,0,262,0
2,2025-01-01,3,21385,25734,20328,5237,6606,0,19270,0,...,-1428,-922,1,-119,-384,-205,62,0,414,0
3,2025-01-01,4,20473,24988,19460,5115,6606,0,19270,0,...,-1423,-915,0,-119,-334,-193,37,0,423,0
4,2025-01-01,5,19634,24912,18654,5080,6606,0,19270,0,...,-1539,-899,0,-119,-363,-622,-282,0,288,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17515,2025-12-31,44,28633,30690,26868,3994,6606,0,22126,0,...,266,-212,912,-345,-145,96,1397,-82,1375,-514
17516,2025-12-31,45,27381,30212,25722,4027,6606,0,22126,0,...,27,-488,831,-432,-254,44,1397,-383,1372,-514
17517,2025-12-31,46,26563,29786,25049,4059,6606,0,22126,0,...,-19,-511,828,-452,-295,38,1368,-402,1370,-514
17518,2025-12-31,47,25481,30558,23955,4115,6606,0,22126,0,...,-1166,-658,622,-452,-383,-431,994,-415,1421,-514


Convert the SETTLEMENT_DATE into timezones

In [6]:
demand["SETTLEMENT_DATE"] = demand["SETTLEMENT_DATE"].dt.tz_localize(tz = "Europe/London")
demand["SETTLEMENT_DATE"].dtypes

datetime64[us, Europe/London]

In [7]:
demand["SETTLEMENT_DATE"] = demand["SETTLEMENT_DATE"].dt.tz_convert("UTC")
demand["time"] = pd.to_timedelta((demand["SETTLEMENT_PERIOD"] - 1) * 30,unit = "m", errors = "coerce")
demand["datetime"] = demand["SETTLEMENT_DATE"] + demand["time"]
demand["datetime"].dtypes

datetime64[us, UTC]

In [8]:
demand

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,time,datetime
0,2025-01-01 00:00:00+00:00,1,21036,26215,19918,5405,6606,0,19270,0,...,1,-232,-386,-558,96,0,241,0,0 days 00:00:00,2025-01-01 00:00:00+00:00
1,2025-01-01 00:00:00+00:00,2,21222,26063,20167,5358,6606,0,19270,0,...,1,-124,-282,-564,51,0,262,0,0 days 00:30:00,2025-01-01 00:30:00+00:00
2,2025-01-01 00:00:00+00:00,3,21385,25734,20328,5237,6606,0,19270,0,...,1,-119,-384,-205,62,0,414,0,0 days 01:00:00,2025-01-01 01:00:00+00:00
3,2025-01-01 00:00:00+00:00,4,20473,24988,19460,5115,6606,0,19270,0,...,0,-119,-334,-193,37,0,423,0,0 days 01:30:00,2025-01-01 01:30:00+00:00
4,2025-01-01 00:00:00+00:00,5,19634,24912,18654,5080,6606,0,19270,0,...,0,-119,-363,-622,-282,0,288,0,0 days 02:00:00,2025-01-01 02:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17515,2025-12-31 00:00:00+00:00,44,28633,30690,26868,3994,6606,0,22126,0,...,912,-345,-145,96,1397,-82,1375,-514,0 days 21:30:00,2025-12-31 21:30:00+00:00
17516,2025-12-31 00:00:00+00:00,45,27381,30212,25722,4027,6606,0,22126,0,...,831,-432,-254,44,1397,-383,1372,-514,0 days 22:00:00,2025-12-31 22:00:00+00:00
17517,2025-12-31 00:00:00+00:00,46,26563,29786,25049,4059,6606,0,22126,0,...,828,-452,-295,38,1368,-402,1370,-514,0 days 22:30:00,2025-12-31 22:30:00+00:00
17518,2025-12-31 00:00:00+00:00,47,25481,30558,23955,4115,6606,0,22126,0,...,622,-452,-383,-431,994,-415,1421,-514,0 days 23:00:00,2025-12-31 23:00:00+00:00


We now have half an hour period of energy demand data.

In [ ]:
demand["SETTLEMENT_DATE"] = demand["SETTLEMENT_DATE"].dt.tz_convert("UTC")
demand["time"] = pd.to_timedelta((demand["SETTLEMENT_PERIOD"] - 1) * 30,unit = "m", errors = "coerce")
demand["datetime"] = demand["SETTLEMENT_DATE"] + demand["time"]
demand["datetime"].dtypes

datetime64[us, UTC]

In [13]:
july_1st = (demand["SETTLEMENT_DATE"] == "2025-06-30 23:00:00+00:00") & (demand["SETTLEMENT_PERIOD"] == 1)
check_july = demand.loc[july_1st].copy()

check_july

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,time,datetime
8686,2025-06-30 23:00:00+00:00,1,23760,27538,22039,978,6606,0,20979,0,...,-545,-297,0,-300,1395,0,-750,-500,0 days,2025-06-30 23:00:00+00:00


In [15]:
daily_grouped = demand.groupby("SETTLEMENT_DATE")

daily_sizes = daily_grouped.size()
daily_sizes.value_counts()

48    363
46      1
50      1
Name: count, dtype: int64

In [16]:
isnt_48  = (daily_sizes != 48)
daily_sizes.loc[isnt_48]

SETTLEMENT_DATE
2025-03-30 00:00:00+00:00    46
2025-10-25 23:00:00+00:00    50
dtype: int64

Continuity Checks

In [18]:
demand.sort_values("datetime")["datetime"].diff().value_counts()

datetime
0 days 00:30:00    17519
Name: count, dtype: int64

In [19]:
demand = demand.drop(columns = ["time"])
demand

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,datetime
0,2025-01-01 00:00:00+00:00,1,21036,26215,19918,5405,6606,0,19270,0,...,-1026,1,-232,-386,-558,96,0,241,0,2025-01-01 00:00:00+00:00
1,2025-01-01 00:00:00+00:00,2,21222,26063,20167,5358,6606,0,19270,0,...,-1023,1,-124,-282,-564,51,0,262,0,2025-01-01 00:30:00+00:00
2,2025-01-01 00:00:00+00:00,3,21385,25734,20328,5237,6606,0,19270,0,...,-922,1,-119,-384,-205,62,0,414,0,2025-01-01 01:00:00+00:00
3,2025-01-01 00:00:00+00:00,4,20473,24988,19460,5115,6606,0,19270,0,...,-915,0,-119,-334,-193,37,0,423,0,2025-01-01 01:30:00+00:00
4,2025-01-01 00:00:00+00:00,5,19634,24912,18654,5080,6606,0,19270,0,...,-899,0,-119,-363,-622,-282,0,288,0,2025-01-01 02:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17515,2025-12-31 00:00:00+00:00,44,28633,30690,26868,3994,6606,0,22126,0,...,-212,912,-345,-145,96,1397,-82,1375,-514,2025-12-31 21:30:00+00:00
17516,2025-12-31 00:00:00+00:00,45,27381,30212,25722,4027,6606,0,22126,0,...,-488,831,-432,-254,44,1397,-383,1372,-514,2025-12-31 22:00:00+00:00
17517,2025-12-31 00:00:00+00:00,46,26563,29786,25049,4059,6606,0,22126,0,...,-511,828,-452,-295,38,1368,-402,1370,-514,2025-12-31 22:30:00+00:00
17518,2025-12-31 00:00:00+00:00,47,25481,30558,23955,4115,6606,0,22126,0,...,-658,622,-452,-383,-431,994,-415,1421,-514,2025-12-31 23:00:00+00:00


In [28]:
demand["London_datetime"] = demand["datetime"].dt.tz_convert("Europe/London")
demand["settlement_day"] = demand["London_datetime"].dt.date

drop = ["ENGLAND_WALES_DEMAND", "NON_BM_STOR","SETTLEMENT_DATE","SCOTTISH_TRANSFER","TSD","RESIDUAL_DEMAND","EMBEDDED_WIND_CAPACITY"]

demand = demand.drop(columns = drop, errors = "ignore")
demand

,SETTLEMENT_PERIOD,ND,EMBEDDED_WIND_GENERATION,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,PUMP_STORAGE_PUMPING,IFA_FLOW,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,...,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,datetime,London_datetime,settlement_day,NET_INTERCONNECTOR,TOTAL_DEMAND
0,1,21036,5405,0,19270,859,-1518,-1026,1,-232,...,-558,96,0,241,0,2025-01-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,2025-01-01,-3382,26441
1,2,21222,5358,0,19270,735,-1513,-1023,1,-124,...,-564,51,0,262,0,2025-01-01 00:30:00+00:00,2025-01-01 00:30:00+00:00,2025-01-01,-3192,26580
2,3,21385,5237,0,19270,691,-1428,-922,1,-119,...,-205,62,0,414,0,2025-01-01 01:00:00+00:00,2025-01-01 01:00:00+00:00,2025-01-01,-2581,26622
3,4,20473,5115,0,19270,931,-1423,-915,0,-119,...,-193,37,0,423,0,2025-01-01 01:30:00+00:00,2025-01-01 01:30:00+00:00,2025-01-01,-2524,25588
4,5,19634,5080,0,19270,854,-1539,-899,0,-119,...,-622,-282,0,288,0,2025-01-01 02:00:00+00:00,2025-01-01 02:00:00+00:00,2025-01-01,-3536,24714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17515,44,28633,3994,0,22126,159,266,-212,912,-345,...,96,1397,-82,1375,-514,2025-12-31 21:30:00+00:00,2025-12-31 21:30:00+00:00,2025-12-31,2748,32627
17516,45,27381,4027,0,22126,160,27,-488,831,-432,...,44,1397,-383,1372,-514,2025-12-31 22:00:00+00:00,2025-12-31 22:00:00+00:00,2025-12-31,1600,31408
17517,46,26563,4059,0,22126,430,-19,-511,828,-452,...,38,1368,-402,1370,-514,2025-12-31 22:30:00+00:00,2025-12-31 22:30:00+00:00,2025-12-31,1411,30622
17518,47,25481,4115,0,22126,458,-1166,-658,622,-452,...,-431,994,-415,1421,-514,2025-12-31 23:00:00+00:00,2025-12-31 23:00:00+00:00,2025-12-31,-982,29596


In [27]:
demand["NET_INTERCONNECTOR"] = demand.filter(regex = "_FLOW$").sum(axis = 1)
demand["TOTAL_DEMAND"] = demand["ND"] + demand["EMBEDDED_WIND_GENERATION"] + demand["EMBEDDED_SOLAR_GENERATION"]
demand

,SETTLEMENT_PERIOD,ND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,PUMP_STORAGE_PUMPING,IFA_FLOW,IFA2_FLOW,BRITNED_FLOW,...,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,datetime,London_datetime,settlement_day,NET_INTERCONNECTOR,TOTAL_DEMAND
0,1,21036,5405,6606,0,19270,859,-1518,-1026,1,...,-558,96,0,241,0,2025-01-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,2025-01-01,-3382,26441
1,2,21222,5358,6606,0,19270,735,-1513,-1023,1,...,-564,51,0,262,0,2025-01-01 00:30:00+00:00,2025-01-01 00:30:00+00:00,2025-01-01,-3192,26580
2,3,21385,5237,6606,0,19270,691,-1428,-922,1,...,-205,62,0,414,0,2025-01-01 01:00:00+00:00,2025-01-01 01:00:00+00:00,2025-01-01,-2581,26622
3,4,20473,5115,6606,0,19270,931,-1423,-915,0,...,-193,37,0,423,0,2025-01-01 01:30:00+00:00,2025-01-01 01:30:00+00:00,2025-01-01,-2524,25588
4,5,19634,5080,6606,0,19270,854,-1539,-899,0,...,-622,-282,0,288,0,2025-01-01 02:00:00+00:00,2025-01-01 02:00:00+00:00,2025-01-01,-3536,24714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17515,44,28633,3994,6606,0,22126,159,266,-212,912,...,96,1397,-82,1375,-514,2025-12-31 21:30:00+00:00,2025-12-31 21:30:00+00:00,2025-12-31,2748,32627
17516,45,27381,4027,6606,0,22126,160,27,-488,831,...,44,1397,-383,1372,-514,2025-12-31 22:00:00+00:00,2025-12-31 22:00:00+00:00,2025-12-31,1600,31408
17517,46,26563,4059,6606,0,22126,430,-19,-511,828,...,38,1368,-402,1370,-514,2025-12-31 22:30:00+00:00,2025-12-31 22:30:00+00:00,2025-12-31,1411,30622
17518,47,25481,4115,6606,0,22126,458,-1166,-658,622,...,-431,994,-415,1421,-514,2025-12-31 23:00:00+00:00,2025-12-31 23:00:00+00:00,2025-12-31,-982,29596


Let's build a helper function that does this automatically

In [33]:
def clean_demand(file_n):
    filename = Path(file_n).stem
    demand = pd.read_csv(file_n)
    formats = ["%Y-%m-%d","%d-%b-%Y","%d-%b-%y"]
    for format_name in formats:
        try:
            parsed = pd.DataFrame()
            parsed["SETTLEMENT_DATE"] = pd.to_datetime(demand["SETTLEMENT_DATE"], format = format_name)
            if parsed["SETTLEMENT_DATE"].isna().sum() == 0:
                demand["SETTLEMENT_DATE"] = parsed["SETTLEMENT_DATE"]
                break
        except(ValueError):
            print(ValueError("Did not match this format, trying next"))
    else:
        raise(ValueError("Did not match with any format, please try again"))
    demand["SETTLEMENT_DATE"] = demand["SETTLEMENT_DATE"].dt.tz_localize(tz = "Europe/London")
    demand["SETTLEMENT_DATE"] = demand["SETTLEMENT_DATE"].dt.tz_convert("UTC")
    demand["time"] = pd.to_timedelta((demand["SETTLEMENT_PERIOD"] - 1) * 30,unit = "m", errors = "coerce")
    demand["datetime"] = demand["SETTLEMENT_DATE"] + demand["time"]
    demand = demand.drop(columns = ["time"])
    demand["London_datetime"] = demand["datetime"].dt.tz_convert("Europe/London")
    demand["settlement_day"] = demand["London_datetime"].dt.date

    drop = ["ENGLAND_WALES_DEMAND", "NON_BM_STOR","SETTLEMENT_DATE","SCOTTISH_TRANSFER","TSD","RESIDUAL_DEMAND","EMBEDDED_WIND_CAPACITY"]

    demand = demand.drop(columns = drop, errors = "ignore")
    demand["NET_INTERCONNECTOR"] = demand.filter(regex = "_FLOW$").sum(axis = 1)
    demand["TOTAL_DEMAND"] = demand["ND"] + demand["EMBEDDED_WIND_GENERATION"] + demand["EMBEDDED_SOLAR_GENERATION"]
    
    demand.to_csv(f"../processed/{filename}_processed.csv", index = False)
    return demand


In [31]:
len(sorted(DEMAND_PATH.glob("demanddata_*.csv")))

4

In [34]:
dataframes = []

for file in sorted(DEMAND_PATH.glob("demanddata_*.csv")):
    df = clean_demand(file)
    dataframes.append(df)


demand_df = pd.concat(dataframes, ignore_index = True)
demand_df = demand_df.sort_values("datetime")

demand_df


Did not match this format, trying next
Did not match this format, trying next
Did not match this format, trying next
Did not match this format, trying next


,SETTLEMENT_PERIOD,ND,EMBEDDED_WIND_GENERATION,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,PUMP_STORAGE_PUMPING,IFA_FLOW,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,...,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,datetime,London_datetime,settlement_day,NET_INTERCONNECTOR,TOTAL_DEMAND
0,1,21940,2713,0,14008,450,-181,-4,586,320,...,-100,210,0,0,0,2022-01-01 00:00:00+00:00,2022-01-01 00:00:00+00:00,2022-01-01,1215,24653
1,2,22427,2755,0,14008,173,-181,-4,583,320,...,-104,176,0,0,0,2022-01-01 00:30:00+00:00,2022-01-01 00:30:00+00:00,2022-01-01,1193,25182
2,3,21896,2930,0,14008,799,-164,-4,567,320,...,-4,-205,0,0,0,2022-01-01 01:00:00+00:00,2022-01-01 01:00:00+00:00,2022-01-01,950,24826
3,4,20693,3090,0,14008,1516,-170,-4,444,320,...,-6,-283,0,0,0,2022-01-01 01:30:00+00:00,2022-01-01 01:30:00+00:00,2022-01-01,688,23783
4,5,19481,3161,0,14008,1490,-753,-4,-383,320,...,-561,-686,0,0,0,2022-01-01 02:00:00+00:00,2022-01-01 02:00:00+00:00,2022-01-01,-1591,22642
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70123,44,28633,3994,0,22126,159,266,-212,912,-345,...,96,1397,-82,1375,-514,2025-12-31 21:30:00+00:00,2025-12-31 21:30:00+00:00,2025-12-31,2748,32627
70124,45,27381,4027,0,22126,160,27,-488,831,-432,...,44,1397,-383,1372,-514,2025-12-31 22:00:00+00:00,2025-12-31 22:00:00+00:00,2025-12-31,1600,31408
70125,46,26563,4059,0,22126,430,-19,-511,828,-452,...,38,1368,-402,1370,-514,2025-12-31 22:30:00+00:00,2025-12-31 22:30:00+00:00,2025-12-31,1411,30622
70126,47,25481,4115,0,22126,458,-1166,-658,622,-452,...,-431,994,-415,1421,-514,2025-12-31 23:00:00+00:00,2025-12-31 23:00:00+00:00,2025-12-31,-982,29596


In [38]:
print(demand_df.sort_values("datetime")["datetime"].diff().value_counts())
print(demand_df["datetime"].duplicated().sum())
print(demand_df.isna().any())
daily_grouped_com = demand_df.groupby("settlement_day")

daily_sizes_com = daily_grouped_com.size()
daily_sizes_com.value_counts()

datetime
0 days 00:30:00    70127
Name: count, dtype: int64
0
SETTLEMENT_PERIOD            False
ND                           False
EMBEDDED_WIND_GENERATION     False
EMBEDDED_SOLAR_GENERATION    False
EMBEDDED_SOLAR_CAPACITY      False
PUMP_STORAGE_PUMPING         False
IFA_FLOW                     False
IFA2_FLOW                    False
BRITNED_FLOW                 False
MOYLE_FLOW                   False
EAST_WEST_FLOW               False
NEMO_FLOW                    False
NSL_FLOW                     False
ELECLINK_FLOW                False
VIKING_FLOW                  False
GREENLINK_FLOW               False
datetime                     False
London_datetime              False
settlement_day               False
NET_INTERCONNECTOR           False
TOTAL_DEMAND                 False
dtype: bool


48    1453
46       4
50       4
Name: count, dtype: int64

In [39]:
copy = demand_df.copy()

copy["datetime"] = copy["datetime"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
copy["London_datetime"] = copy["London_datetime"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
copy["settlement_day"] = copy["settlement_day"].astype(str)

copy

,SETTLEMENT_PERIOD,ND,EMBEDDED_WIND_GENERATION,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,PUMP_STORAGE_PUMPING,IFA_FLOW,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,...,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,datetime,London_datetime,settlement_day,NET_INTERCONNECTOR,TOTAL_DEMAND
0,1,21940,2713,0,14008,450,-181,-4,586,320,...,-100,210,0,0,0,2022-01-01T00:00:00+0000,2022-01-01T00:00:00+0000,2022-01-01,1215,24653
1,2,22427,2755,0,14008,173,-181,-4,583,320,...,-104,176,0,0,0,2022-01-01T00:30:00+0000,2022-01-01T00:30:00+0000,2022-01-01,1193,25182
2,3,21896,2930,0,14008,799,-164,-4,567,320,...,-4,-205,0,0,0,2022-01-01T01:00:00+0000,2022-01-01T01:00:00+0000,2022-01-01,950,24826
3,4,20693,3090,0,14008,1516,-170,-4,444,320,...,-6,-283,0,0,0,2022-01-01T01:30:00+0000,2022-01-01T01:30:00+0000,2022-01-01,688,23783
4,5,19481,3161,0,14008,1490,-753,-4,-383,320,...,-561,-686,0,0,0,2022-01-01T02:00:00+0000,2022-01-01T02:00:00+0000,2022-01-01,-1591,22642
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70123,44,28633,3994,0,22126,159,266,-212,912,-345,...,96,1397,-82,1375,-514,2025-12-31T21:30:00+0000,2025-12-31T21:30:00+0000,2025-12-31,2748,32627
70124,45,27381,4027,0,22126,160,27,-488,831,-432,...,44,1397,-383,1372,-514,2025-12-31T22:00:00+0000,2025-12-31T22:00:00+0000,2025-12-31,1600,31408
70125,46,26563,4059,0,22126,430,-19,-511,828,-452,...,38,1368,-402,1370,-514,2025-12-31T22:30:00+0000,2025-12-31T22:30:00+0000,2025-12-31,1411,30622
70126,47,25481,4115,0,22126,458,-1166,-658,622,-452,...,-431,994,-415,1421,-514,2025-12-31T23:00:00+0000,2025-12-31T23:00:00+0000,2025-12-31,-982,29596


In [40]:
import sqlite3

DB_PATH = "../processed/energy.db"

con = sqlite3.connect(DB_PATH)

copy.to_sql("demand",con, if_exists="replace", index = False)

query = """
    CREATE INDEX IF NOT EXISTS idx_demand_datetime ON demand (datetime)
"""
con.execute(query)
con.commit()


Check: fetch the data from the database and put into a dataframe and observe

In [51]:
test_query = """ SELECT name FROM sqlite_master WHERE type = 'index' AND tbl_name = 'demand'"""
test = pd.read_sql(test_query, con)
test

,name
0,idx_demand_datetime


In [52]:
test_query2 = """ SELECT COUNT(*), MIN(datetime), MAX(datetime) FROM demand"""
cursor = con.execute(test_query2)
cursor.fetchone()

(70128, '2022-01-01T00:00:00+0000', '2025-12-31T23:30:00+0000')

In [53]:
con.close()

Make the load function

In [55]:
def load_demand(db_path):
    con = sqlite3.connect(db_path)
    load_query = """ SELECT * FROM demand ORDER BY datetime"""
    load = pd.read_sql(load_query, con)
    con.close()
    load["datetime"] = pd.to_datetime(load["datetime"],format = "%Y-%m-%dT%H:%M:%S%z", utc = True)
    load["London_datetime"] = load["datetime"].dt.tz_convert("Europe/London")
    load["settlement_day"] = load["London_datetime"].dt.date
    return load
    


Lets run checks

In [56]:
load = load_demand(DB_PATH)
print(load["datetime"].dtypes)
print(load["London_datetime"].dtypes)
print(load["settlement_day"].dtypes)

datetime64[us, UTC]
datetime64[us, Europe/London]
object


In [58]:
load

,SETTLEMENT_PERIOD,ND,EMBEDDED_WIND_GENERATION,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,PUMP_STORAGE_PUMPING,IFA_FLOW,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,...,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,datetime,London_datetime,settlement_day,NET_INTERCONNECTOR,TOTAL_DEMAND
0,1,21940,2713,0,14008,450,-181,-4,586,320,...,-100,210,0,0,0,2022-01-01 00:00:00+00:00,2022-01-01 00:00:00+00:00,2022-01-01,1215,24653
1,2,22427,2755,0,14008,173,-181,-4,583,320,...,-104,176,0,0,0,2022-01-01 00:30:00+00:00,2022-01-01 00:30:00+00:00,2022-01-01,1193,25182
2,3,21896,2930,0,14008,799,-164,-4,567,320,...,-4,-205,0,0,0,2022-01-01 01:00:00+00:00,2022-01-01 01:00:00+00:00,2022-01-01,950,24826
3,4,20693,3090,0,14008,1516,-170,-4,444,320,...,-6,-283,0,0,0,2022-01-01 01:30:00+00:00,2022-01-01 01:30:00+00:00,2022-01-01,688,23783
4,5,19481,3161,0,14008,1490,-753,-4,-383,320,...,-561,-686,0,0,0,2022-01-01 02:00:00+00:00,2022-01-01 02:00:00+00:00,2022-01-01,-1591,22642
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70123,44,28633,3994,0,22126,159,266,-212,912,-345,...,96,1397,-82,1375,-514,2025-12-31 21:30:00+00:00,2025-12-31 21:30:00+00:00,2025-12-31,2748,32627
70124,45,27381,4027,0,22126,160,27,-488,831,-432,...,44,1397,-383,1372,-514,2025-12-31 22:00:00+00:00,2025-12-31 22:00:00+00:00,2025-12-31,1600,31408
70125,46,26563,4059,0,22126,430,-19,-511,828,-452,...,38,1368,-402,1370,-514,2025-12-31 22:30:00+00:00,2025-12-31 22:30:00+00:00,2025-12-31,1411,30622
70126,47,25481,4115,0,22126,458,-1166,-658,622,-452,...,-431,994,-415,1421,-514,2025-12-31 23:00:00+00:00,2025-12-31 23:00:00+00:00,2025-12-31,-982,29596


In [62]:
import datetime

july_1st_test = (load["settlement_day"] == datetime.date(2025, 7, 1)) & (load["SETTLEMENT_PERIOD"] == 1)
check_july_test = load.loc[july_1st_test].copy()

check_july_test

,SETTLEMENT_PERIOD,ND,EMBEDDED_WIND_GENERATION,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,PUMP_STORAGE_PUMPING,IFA_FLOW,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,...,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,datetime,London_datetime,settlement_day,NET_INTERCONNECTOR,TOTAL_DEMAND
61294,1,23760,978,0,20979,60,-420,-406,-545,-297,...,-300,1395,0,-750,-500,2025-06-30 23:00:00+00:00,2025-07-01 00:00:00+01:00,2025-07-01,-1823,24738


In [63]:
print(load["datetime"].equals(demand_df["datetime"]))
print(load["TOTAL_DEMAND"].equals(demand_df["TOTAL_DEMAND"]))
print(load["settlement_day"].equals(demand_df["settlement_day"]))

True
True
True
